In [ ]:
# 訓練データとテストデータの画像を読み込む
# （サイズは縦横224pxにリサイズする）
import tensorflow as tf

train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "dog_cat_photos/train",
    image_size=(224, 224),
    label_mode="binary",
    batch_size=200,
    shuffle=True
)

test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "dog_cat_photos/test",
    image_size=(224, 224),
    label_mode="binary",
    batch_size=75,
    shuffle=False
)


In [ ]:
# 分類名（dog／cat）をリストとして格納する
class_names = train_dataset.class_names
class_names

In [ ]:
# 画像の水増しをする関数の定義
def flip_left_right(image, label):   # 左右反転
    image = tf.image.flip_left_right(image)
    return image, label

def flip_up_down(image, label):      # 上下反転
    image = tf.image.flip_up_down(image)
    return image, label

def rot90(image, label):             # 反時計回りに90度回転
    image = tf.image.rot90(image)
    return image, label

def rot180(image, label):            # 反時計回りに180度回転
    image = tf.image.rot90(image, k=2)
    return image, label

def rot270(image, label):            # 反時計回りに270度回転
    image = tf.image.rot90(image, k=3)
    return image, label


# 画像の水増し処理の実行
train_dataset_lr     = train_dataset.map(flip_left_right)
train_dataset_ud     = train_dataset.map(flip_up_down)
train_dataset_rot90  = train_dataset.map(rot90)
train_dataset_rot180 = train_dataset.map(rot180)
train_dataset_rot270 = train_dataset.map(rot270)


# 水増ししたデータを訓練データに追加する
train_dataset = train_dataset.concatenate(train_dataset_lr)
train_dataset = train_dataset.concatenate(train_dataset_ud)
train_dataset = train_dataset.concatenate(train_dataset_rot90)
train_dataset = train_dataset.concatenate(train_dataset_rot180)
train_dataset = train_dataset.concatenate(train_dataset_rot270)

In [ ]:
# データをシャッフルする
train_dataset = train_dataset.shuffle(200)

In [4]:
# MobileNetV2モデルを作成する
input_layer = tf.keras.Input(shape=(224, 224, 3))   # 入力層
l_layer = tf.keras.applications.mobilenet_v2.preprocess_input(input_layer)   # 前処理（正規化）をする層

base_model = tf.keras.applications.mobilenet_v2.MobileNetV2(
    input_shape=(224, 224, 3),
    input_tensor=l_layer,
    include_top=False,
    weights="imagenet",
    pooling='avg'
)
base_model.trainable = False


# Dense層を追加する
output_layer = tf.keras.layers.Dense(1, activation='sigmoid')


# base_modelに先ほどのDense層を追加したモデルを作成する
model = tf.keras.Sequential([
    base_model,
    output_layer
])


# modelをcompileする
model.compile(optimizer="adam",
              loss='binary_crossentropy',
              metrics=["accuracy"])


# modelに学習させる
model.fit(train_dataset, epochs=20)

Epoch 1/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 14s 2s/step - accuracy: 0.4933 - loss: 0.7696
Epoch 2/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 807ms/step - accuracy: 0.6767 - loss: 0.6281
Epoch 3/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 950ms/step - accuracy: 0.7900 - loss: 0.5128
Epoch 4/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 868ms/step - accuracy: 0.8633 - loss: 0.4213
Epoch 5/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 881ms/step - accuracy: 0.8967 - loss: 0.3482
Epoch 6/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 929ms/step - accuracy: 0.9200 - loss: 0.2915
Epoch 7/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 881ms/step - accuracy: 0.9367 - loss: 0.2471
Epoch 8/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 915ms/step - accuracy: 0.9433 - loss: 0.2126
Epoch 9/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 872ms/step - accuracy: 0.9600 - loss: 0.1858
Epoch 10/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 874ms/step - accuracy: 0.9700 - loss: 0.1639
Epoch 11/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 857ms/step - accuracy: 0.9800 - loss: 0.1472
Epoch 12/20
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 886ms/step - accuracy: 0.9867 - loss

In [6]:
# テストデータで分類を実行する
pred_data = model.predict(test_dataset)

# 分類した結果を確認する
pred_data

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 895ms/step


array([[0.00683863],
       [0.06096942],
       [0.01580435],
       [0.01884561],
       [0.02124981],
       [0.00554318],
       [0.01216285],
       [0.01441504],
       [0.0020196 ],
       [0.03443282],
       [0.01003736],
       [0.00650636],
       [0.02270691],
       [0.03445838],
       [0.00321872],
       [0.00463269],
       [0.00809631],
       [0.00570948],
       [0.02795001],
       [0.0509618 ],
       [0.26319194],
       [0.10488679],
       [0.01616058],
       [0.01270619],
       [0.5740829 ],
       [0.02826088],
       [0.04216414],
       [0.0398855 ],
       [0.09793741],
       [0.00385029],
       [0.0273762 ],
       [0.86884755],
       [0.01960538],
       [0.03314788],
       [0.05004925],
       [0.79952717],
       [0.08444372],
       [0.0593729 ],
       [0.07551619],
       [0.06323908],
       [0.0076937 ],
       [0.01411119],
       [0.05359409],
       [0.01780791],
       [0.0145547 ],
       [0.07964648],
       [0.00492207],
       [0.063

In [7]:
# evaluate()でモデルの性能を評価する
model.evaluate(test_dataset)

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.9700 - loss: 0.0870


[0.08700992912054062, 0.9700000286102295]